In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_config

In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_helpers

## 1. Read Silver Orders + All Dimensions

In [0]:
log("Reading Silver orders and Gold dimensions ...")

df_orders      = spark.table(TBL_SILVER_ORDERS)
df_customer    = spark.table(TBL_GOLD_DIM_CUSTOMER)
df_product     = spark.table(TBL_GOLD_DIM_PRODUCT)
df_date        = spark.table(TBL_GOLD_DIM_DATE)
df_location    = spark.table(TBL_GOLD_DIM_LOCATION)
df_shipment    = spark.table(TBL_GOLD_DIM_SHIPMENT)

## 2. Join to Resolve Dimension Keys

In [0]:
df_fact = (
    # join dim_customer
    df_orders
    .join(
        df_customer.select("customer_id"),
        on="customer_id",
        how="left" 
    )
    # join dim_product
    .join(
        df_product.select("product_id"),
        on="product_id",
        how="left" 
    )
    # join dim_date on order_date
    .join(
        df_date.select(
            F.col("date").alias("order_date"),
            F.col("date_id").alias("order_date_id")
        ),
        on="order_date",
        how="left" 
    )
    # join dim_date on ship_date
    .join(
        df_date.select(
            F.col("date").alias("ship_date"),
            F.col("date_id").alias("ship_date_id")
        ),
        on="ship_date",
        how="left" 
    )        
    # join dim_location
    .join(
        df_location.select("location_id", "city", "state", "postal_code"),
        on=["city", "state", "postal_code"],
        how="left"
    )
    # join dim_shipment
    .join(
        df_shipment.select("shipment_id", "ship_mode"),
        on="ship_mode",
        how="left"
    )
)

## 3. Select Final Fact Columns

In [0]:
df_fact_orders = df_fact.select(
    F.col("row_id"),
    F.col("order_id"),
    F.col("customer_id"),
    F.col("product_id"),
    F.col("order_date_id"),
    F.col("ship_date_id"),
    F.col("location_id"),
    F.col("shipment_id"),
    F.col("sales"),
    F.col("quantity"),
    F.col("discount"),
    F.col("profit")
)

log(f"fact_orders row count: {df_fact_orders.count():,}")

## 4. Write/Upsert to Gold

In [0]:
if spark.catalog.tableExists(TBL_GOLD_FACT_ORDERS):
    log(f"Updating into {TBL_GOLD_FACT_ORDERS} ...")

    delta_table = DeltaTable.forName(spark, TBL_GOLD_FACT_ORDERS)
    (
        delta_table.alias("target")
        .merge(
            df_fact_orders.alias("source"),
            "target.row_id = source.row_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    log(f"✅ Done. Table {TBL_GOLD_FACT_ORDERS} upserted")

else:
    log(f"Creating {TBL_GOLD_FACT_ORDERS} ...")
    (
        df_fact_orders.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TBL_GOLD_FACT_ORDERS)
    )

    log(f"✅ Done. Table {TBL_GOLD_FACT_ORDERS} created")

In [0]:
display(spark.table(TBL_GOLD_FACT_ORDERS).orderBy("row_id").limit(10))

In [0]:
# Summary stats
spark.sql(f"""
    SELECT
        COUNT(*)            AS total_orders,
        ROUND(SUM(sales), 2)   AS total_sales,
        ROUND(SUM(profit), 2)  AS total_profit,
        ROUND(AVG(discount), 4) AS avg_discount
    FROM {TBL_GOLD_FACT_ORDERS}
""").show()